# LGG MRI Segmentation — project walkthrough & pilot viewer (read-only)

This notebook **does not launch training, does not touch the test set, and creates no AWS resources**. It explains the code/data flow and shows the existing pilot job `ajay-mlp1-pilot-bce3ep-20261108-0315`.

Data flow: Drive dataset (manifests + TIFFs) → hash-verified → `s3://ajay-mlp1-lgg-044450716277-use1/data/trainval/trainval.tar` (train+val only) → SageMaker training job (official PyTorch 2.6 GPU container, `entry.py` → `scripts/train.py`) → `output/<job>/output/model.tar.gz`. Test data is in a separate prefix. The TRAINING role cannot read it. NOTE (2026-11-08 03:49 EST): at the user's explicit request the Studio role has AdministratorAccess and its test-data deny was removed, so Studio CAN technically read the test prefix — by protocol it must not be read and is only used later by `scripts/evaluate_test.py` after `scripts/freeze.py`.

In [ ]:
import json, os, tarfile, boto3
REGION='us-east-1'; BUCKET='ajay-mlp1-lgg-044450716277-use1'
JOB='ajay-mlp1-pilot-bce3ep-20261108-0315'
sm=boto3.client('sagemaker',region_name=REGION); s3=boto3.client('s3',region_name=REGION); logs=boto3.client('logs',region_name=REGION)

## 1. Job status (read-only DescribeTrainingJob)

In [ ]:
j=sm.describe_training_job(TrainingJobName=JOB)
print(j['TrainingJobStatus'], j['SecondaryStatus'], 'billable s =', j.get('BillableTimeInSeconds'))
for t in j['SecondaryStatusTransitions']: print(t['StartTime'], t['Status'], '-', t.get('StatusMessage'))
print('Console:', f'https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{JOB}')

## 2. Training logs (CloudWatch)

In [ ]:
streams=logs.describe_log_streams(logGroupName='/aws/sagemaker/TrainingJobs',logStreamNamePrefix=JOB)['logStreams']
for s in streams:
    for e in logs.get_log_events(logGroupName='/aws/sagemaker/TrainingJobs',logStreamName=s['logStreamName'],startFromHead=True)['events'][-60:]:
        print(e['message'])

## 3. Pilot artifacts (environment, config, validation-only metrics)
PILOT = 3 epochs, plain BCE, seed 0: a pipeline/throughput check, **not** a research result.

In [ ]:
PO=os.path.expanduser('~/brain-mri-segmentation/runs/managed_pilot'); os.makedirs(PO,exist_ok=True)
s3.download_file(BUCKET, f'output/{JOB}/output/model.tar.gz', PO+'/model.tar.gz')
tarfile.open(PO+'/model.tar.gz').extractall(PO)
print(json.dumps(json.load(open(PO+'/environment.json')),indent=1))
m=json.load(open(PO+'/runs/PILOT_bce_3ep_s0/run_meta.json'))
print('threshold', m['selected_threshold'], 'params', m['params'], 'train_time_s', round(m['train_time_s'],1))
print(json.dumps(m['val_at_selected'],indent=1))

In [ ]:
for r in json.load(open(PO+'/runs/PILOT_bce_3ep_s0/train_log.json')):
    print(r['epoch'], round(r['train_loss'],4), round(r['val@0.5']['pooled_foreground_dice'],4), round(r['elapsed_s'],1))

## 4. Code map
- `src/lggseg/data.py` manifest loader, channel-preserving TIFF read, train-only normalization, aligned aug
- `src/lggseg/model.py` U-Net from scratch; `losses.py` BCE / BCE+Dice / diagnostics; `metrics.py` course conventions
- `scripts/train.py` val-only checkpoint/threshold; `scripts/freeze.py` + `scripts/evaluate_test.py` one-time frozen test protocol
- `sagemaker/entry.py` container entry point

**Launching new jobs is intentionally not in this notebook**; it requires explicit user approval of cost and run plan.